In [1]:
from collections import Counter

# 1. vocabulary 만드는 과정

In [11]:
# BPE는 먼저 학습 데이터에서 자주 등장하는 문자 조합을 찾는다.

corpus = [
    "low",
    "lower",
    "lowest",
    "low",
    "lower"
]

# 실제 tokenizer는 byte-level 등 여러 방식이 있지만, 여기서는 BPE 원리를 보기 위해 문자 단위에서 시작
words = [list(word) for word in corpus]
words

[['l', 'o', 'w'],
 ['l', 'o', 'w', 'e', 'r'],
 ['l', 'o', 'w', 'e', 's', 't'],
 ['l', 'o', 'w'],
 ['l', 'o', 'w', 'e', 'r']]

In [12]:
# 초기 vocabulary 생성
# 처음에는 l, o, w, e, r, s, t가 token

vocabulary = {}

for word in words:
    for token in word:
        if token not in vocabulary:
            vocabulary[token] = len(vocabulary)

print(">>> 초기 vocabulary\n", vocabulary)

>>> 초기 vocabulary
 {'l': 0, 'o': 1, 'w': 2, 'e': 3, 'r': 4, 's': 5, 't': 6}


In [13]:
# BPE 특징, 가장 자주 등장하는 Token Pair 찾기
# 모든 인접한 pair의 등장 횟수를 카운트

def get_pair_frequency(words):
    pair_frequency = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_frequency[pair] += 1

    return pair_frequency


pair_frequency = get_pair_frequency(words)

print(">>> Token Pair 빈도")
for pair, count in pair_frequency.most_common():
    print(f"{pair} : {count}회")


>>> Token Pair 빈도
('l', 'o') : 5회
('o', 'w') : 5회
('w', 'e') : 3회
('e', 'r') : 2회
('e', 's') : 1회
('s', 't') : 1회


In [14]:
# 가장 많이 등장하는 Pair를 Merge
def merge_pair(words, target_pair):
    new_words = []

    for word in words:
        new_word = []
        i = 0

        while i < len(word):

            # 현재 token과 다음 token이 target pair라면 두 token을 하나로 합친다.
            if (i < len(word) - 1 and (word[i], word[i + 1]) == target_pair):
                merged_token = word[i] + word[i + 1]
                new_word.append(merged_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        new_words.append(new_word)

    return new_words


num_merges = 4  # merge 회수

for step in range(num_merges):

    pair_frequency = get_pair_frequency(words) # 현재 상태에서 가장 많이 등장하는 pair 찾기

    if not pair_frequency:
        break

    best_pair = pair_frequency.most_common(1)[0][0]

    new_token = "".join(best_pair)

    print(f"\n--- Merge {step + 1} ---")
    print(f"선택된 Pair : {best_pair}")
    print(f"새로운 Token : '{new_token}'")

    words = merge_pair(words, best_pair) # Pair를 하나의 Token으로 merge

    # 새로운 Token을 Vocabulary에 추가
    if new_token not in vocabulary:
        vocabulary[new_token] = len(vocabulary)

    print(">>> 현재 상태:")
    for word in words:
        print(word)


--- Merge 1 ---
선택된 Pair : ('l', 'o')
새로운 Token : 'lo'
>>> 현재 상태:
['lo', 'w']
['lo', 'w', 'e', 'r']
['lo', 'w', 'e', 's', 't']
['lo', 'w']
['lo', 'w', 'e', 'r']

--- Merge 2 ---
선택된 Pair : ('lo', 'w')
새로운 Token : 'low'
>>> 현재 상태:
['low']
['low', 'e', 'r']
['low', 'e', 's', 't']
['low']
['low', 'e', 'r']

--- Merge 3 ---
선택된 Pair : ('low', 'e')
새로운 Token : 'lowe'
>>> 현재 상태:
['low']
['lowe', 'r']
['lowe', 's', 't']
['low']
['lowe', 'r']

--- Merge 4 ---
선택된 Pair : ('lowe', 'r')
새로운 Token : 'lower'
>>> 현재 상태:
['low']
['lower']
['lowe', 's', 't']
['low']
['lower']


In [15]:
print(">>> vocabulary\n", vocabulary)

>>> vocabulary
 {'l': 0, 'o': 1, 'w': 2, 'e': 3, 'r': 4, 's': 5, 't': 6, 'lo': 7, 'low': 8, 'lowe': 9, 'lower': 10}


In [18]:
pip install tokenizers

In [19]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [38]:
# 1. Training Data

training_data = """I studied artificial intelligence before.
When I studied artificial intelligence before, the concepts were very unfamiliar and difficult.
Now, there are many more resources and articles that explain things in simple ways.
Now, I am studying artificial intelligence again.
"""

with open("sample-data.txt", "w", encoding="utf-8") as f:
    f.write(training_data)

In [39]:
# 2. BPE Tokenizer 생성

tokenizer = Tokenizer(BPE(unk_token="[UNK]")) # 모르는 토큰은 UNK (Unknown) 처리
tokenizer.pre_tokenizer = Whitespace() # 공백 기준으로 초기 Token 설정

In [40]:
# 3. BPE Trainer 설정 및 Training Data로 BPE 학습

trainer = BpeTrainer(vocab_size=100, special_tokens=["[UNK]"])
tokenizer.train_from_iterator(training_data.strip().split("\n"), trainer=trainer)

In [41]:
# 4. Vocabulary
print(">>> Vocabulary")
for token, token_id in sorted(tokenizer.get_vocab().items(), key=lambda x: x[1]):
    print(f"{token:20} → {token_id}")

print(tokenizer.encode("I studied artificial intelligence before.").tokens)

>>> Vocabulary
[UNK]                → 0
,                    → 1
.                    → 2
I                    → 3
N                    → 4
W                    → 5
a                    → 6
b                    → 7
c                    → 8
d                    → 9
e                    → 10
f                    → 11
g                    → 12
h                    → 13
i                    → 14
l                    → 15
m                    → 16
n                    → 17
o                    → 18
p                    → 19
r                    → 20
s                    → 21
t                    → 22
u                    → 23
v                    → 24
w                    → 25
x                    → 26
y                    → 27
in                   → 28
re                   → 29
ar                   → 30
ce                   → 31
ic                   → 32
en                   → 33
fic                  → 34
li                   → 35
th                   → 36
art                  → 37
al     

In [42]:
# 5. 문장 입력 시 Token/Token ID
sentence = "I studied artificial intelligence again."
output = tokenizer.encode(sentence)

print(">>> Encoding 결과")
print("tokens :", output.tokens)
print("ids    :", output.ids)

print("\n토큰 → ID 대응")
for token, token_id in zip(output.tokens, output.ids):
    print(f"{token:15} → {token_id}")

>>> Encoding 결과
tokens : ['I', 'studied', 'artificial', 'intelligence', 'ag', 'ain', '.']
ids    : [3, 66, 53, 54, 70, 57, 2]

토큰 → ID 대응
I               → 3
studied         → 66
artificial      → 53
intelligence    → 54
ag              → 70
ain             → 57
.               → 2


In [43]:
# 5. 잘 모르는 문장 입력 시 Token/Token ID
new_sentence = "Studying intelligent articles is difficult."
output = tokenizer.encode(new_sentence)

for token, token_id in zip(output.tokens, output.ids):
    print(f"{token:15} → {token_id}")

[UNK]           → 0
t               → 22
u               → 23
dy              → 75
ing             → 63
intelligen      → 52
t               → 22
art             → 37
ic              → 32
le              → 62
s               → 21
i               → 14
s               → 21
di              → 40
ffic            → 78
ult             → 95
.               → 2


In [45]:
# 6. 디코딩

ids = tokenizer.encode("I studied artificial intelligence again.").ids
print(tokenizer.decode(ids))

I studied artificial intelligence ag ain .


In [46]:
# 7. offsets로 원문 위치 확인

output = tokenizer.encode("I studied artificial intelligence again.")
for token, offset in zip(output.tokens, output.offsets):
    print(f"{token:15} {offset}")

I               (0, 1)
studied         (2, 9)
artificial      (10, 20)
intelligence    (21, 33)
ag              (34, 36)
ain             (36, 39)
.               (39, 40)


# 3. 한국어도?

In [47]:
# 1. 일반 문장 데이터 준비
corpus = [
    "나의 인공지능 모델입니다.",
    "나의 축구공입니다.",
    "인공지능 모델을 공부합니다."
]

# 2. BPE 토크나이저 설정
tokenizer = Tokenizer(BPE(unk_token="[UNK]")) #학습 데이터에서 한 번도 본 적 없는 문자는 [UNK](Unknown)라는 특수 토큰으로 대체하여 처리

# [핵심] 컴퓨터가 스스로 띄어쓰기를 인식하고, 
# 단어 뒤에 특수 기호를 자동으로 붙이도록 명령하는 내장 기능입니다.
tokenizer.pre_tokenizer = Whitespace() # 사전 토크나이저 설정, 공백(Whitespace)과 줄바꿈을 기준으로 쪼개라

# 3. 모델 학습 (단어 사전 크기를 50개로 설정)
trainer = BpeTrainer(vocab_size=50, special_tokens=["[UNK]"])
tokenizer.train_from_iterator(corpus, trainer)

# 4. 검증: 일반 문장을 넣어서 어떻게 쪼개지는지 확인하기
sentence = "나의 인공지능 모델"
encoded = tokenizer.encode(sentence)

print("🧩 사람이 입력한 문장:", sentence)
print("🤖 컴퓨터가 내부적으로 분리한 결과 (Tokens):", encoded.tokens)


🧩 사람이 입력한 문장: 나의 인공지능 모델
🤖 컴퓨터가 내부적으로 분리한 결과 (Tokens): ['나의', '인공지능', '모델']
